In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Load Dataset (from Lab 9/10)

In [3]:
df = pd.read_csv("data.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (4600, 18)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,5/2/2014 0:00,313000.0,3.0,1.50,1340.0,7912.0,1.5,0.0,0.0,3,1340.0,0.0,1955.0,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,5/2/2014 0:00,2384000.0,5.0,2.50,3650.0,9050.0,2.0,0.0,4.0,5,3370.0,280.0,1921.0,0,709 W Blaine St,Seattle,WA 98119,USA
2,5/2/2014 0:00,342000.0,3.0,2.00,1930.0,11947.0,1.0,0.0,0.0,4,1930.0,0.0,1966.0,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,5/2/2014 0:00,420000.0,3.0,2.25,2000.0,8030.0,1.0,0.0,0.0,4,1000.0,1000.0,1963.0,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,5/2/2014 0:00,550000.0,4.0,2.50,1940.0,10500.0,1.0,0.0,0.0,4,1140.0,800.0,1976.0,1992,9105 170th Ave NE,Redmond,WA 98052,USA


# Data Pre-Processing (as done in Lab 10)

In [4]:
# 1. Handle missing values
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "Unknown", inplace=True)

# 2. Convert numeric columns to integer (nullable Int64)
float_cols = df.select_dtypes(include=['float64']).columns
for col in float_cols:
    df[col] = df[col].round().astype('Int64')

# 3. Convert date column to datetime (we will drop it later)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print("Missing values after handling:", df.isnull().sum().sum())

Missing values after handling: 51


# Feature Engineering: Encode Categorical Variables

In [5]:
# Identify categorical columns (object type)
cat_cols = df.select_dtypes(include=['object']).columns
print("Categorical columns found:", list(cat_cols))

# Apply LabelEncoder to each categorical column
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

print("All categorical columns have been encoded to numbers.")
df.head()

Categorical columns found: ['street', 'city', 'statezip', 'country']
All categorical columns have been encoded to numbers.


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02,313000,3,2,1340,7912,2,0,0,3,1340,0,1955,2005,1522,36,62,0
1,2014-05-02,2384000,5,2,3650,9050,2,0,4,5,3370,280,1921,0,3898,35,58,0
2,2014-05-02,342000,3,2,1930,11947,1,0,0,4,1930,0,1966,0,2290,18,26,0
3,2014-05-02,420000,3,2,2000,8030,1,0,0,4,1000,1000,1963,0,4261,3,7,0
4,2014-05-02,550000,4,2,1940,10500,1,0,0,4,1140,800,1976,1992,4349,31,31,0


# Define Features (X) and Target (y)
## Drop the 'date' column because it's datetime and not needed for prediction

In [6]:
# Use 'price' as the target (what we want to predict)
target = 'price'

# Drop the 'date' column (it's datetime and cannot be used directly)
X = df.drop(columns=[target, 'date'])
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nAll features are now numeric:")
print(X.dtypes.value_counts())

Features shape: (4600, 16)
Target shape: (4600,)

All features are now numeric:
Int64    10
int64     6
Name: count, dtype: int64


# Train-Test Split (80% training, 20% testing)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

Training samples: 3680
Testing samples:  920


# Choose and Apply Model – Linear Regression

In [8]:
# --- Clean training data before fitting ---
print("NaN in X_train before fix:", X_train.isnull().sum().sum())
print("NaN in y_train before fix:", y_train.isnull().sum())

# Fill NaN with median (or 0 if median fails)
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())
y_train = y_train.fillna(y_train.median())

# Drop any column that is still all NaN (should not happen after fill)
X_train = X_train.dropna(axis=1, how='all')
X_test = X_test[X_train.columns]

print("After fix - NaN in X_train:", X_train.isnull().sum().sum())
print("After fix - NaN in y_train:", y_train.isnull().sum())

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)
print("Model training completed successfully!")

NaN in X_train before fix: 26
NaN in y_train before fix: 3
After fix - NaN in X_train: 0
After fix - NaN in y_train: 0
Model training completed successfully!


# Testing / Predicting

In [9]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Show first 10 actual vs predicted prices
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
print("First 10 predictions:")
results.head(10)

First 10 predictions:


,Actual,Predicted
0,544000,3.022044e+05
1,0,3.107165e+05
2,1712500,1.060285e+06
3,365000,5.491688e+05
4,275000,3.707715e+05
5,625000,5.820530e+05
6,453000,4.592658e+05
7,300000,4.107914e+05
8,417986,4.950570e+05
9,672500,5.342165e+05


# Display Accuracy Score (R² Score)

In [10]:
# R² score measures how well the predictions match the actual values
r2 = r2_score(y_test, y_pred)
print(f"R² Score (Accuracy for regression): {r2:.4f}")

# Mean Squared Error and Root Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: ${rmse:.2f}")

R² Score (Accuracy for regression): 0.0324
Mean Squared Error: 986754796556.42
Root Mean Squared Error: $993355.32


# Summary

In [11]:
print("\n" + "="*50)
print("LAB 11 - MODEL EVALUATION SUMMARY")
print("="*50)
print(f"Model used: Linear Regression")
print(f"R² Score (accuracy): {r2:.4f}")
print(f"Mean Squared Error: {mse:.2f}")
print(f"RMSE: ${rmse:.2f}")
print("="*50)


LAB 11 - MODEL EVALUATION SUMMARY
Model used: Linear Regression
R² Score (accuracy): 0.0324
Mean Squared Error: 986754796556.42
RMSE: $993355.32
